##### Copyright 2026 The MediaPipe Authors. All Rights Reserved.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<a href="https://colab.research.google.com/github/google-ai-edge/mediapipe-samples/blob/main/examples/text_to_speech/python/text_to_speech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text to Speech with MediaPipe Tasks

In this notebook you will use the MediaPipe Tasks `TextToSpeech` API to turn text into natural-sounding speech entirely on device.

The sample uses [Kokoro 82M](https://huggingface.co/litert-community/Kokoro-82M), a small multilingual text-to-speech model, converted to the LiteRT-LM format. You will:

1. Synthesize a sentence in a single call, play it back, save it as a `.wav` file, and inspect its word timestamps.
2. Stream the audio clause by clause, so that playback can start before the whole text has been synthesized.
3. Push text incrementally, as you would when speaking the output of a language model while it is being generated.

## Preparation

Let's start with installing MediaPipe. The `TextToSpeech` task requires `mediapipe` 1.2.0 or newer.

In [ ]:
%pip install -q "mediapipe>=1.2.0" numpy

Then download an off-the-shelf model. This example uses Kokoro 82M (~295 MB, [Apache 2.0](https://www.apache.org/licenses/LICENSE-2.0)), which bundles 54 voices in 9 languages and produces 24 kHz audio. The file is only downloaded if it isn't already present.

In [ ]:
import os
import urllib.request

model_url = 'https://huggingface.co/litert-community/Kokoro-82M/resolve/main/kokoro_int8_gpu_fp16.litertlm'
model_path = model_url.split('/')[-1]

if not os.path.exists(model_path):
  print(f'Downloading {model_path}...')
  urllib.request.urlretrieve(model_url, model_path)
print(f'Model: {model_path} ({os.path.getsize(model_path) / 1e6:.0f} MB)')

## Creating the TextToSpeech task

Configure the task with `TextToSpeechOptions`:

- `base_options`: the model path and the delegate (hardware) to run on. `Delegate.GPU` runs the model on the GPU where one is available; `Delegate.CPU` runs it on the CPU. If the GPU can't be initialized in your runtime, the code below falls back to the CPU.
- `default_voice_id`: the voice used when you don't pass one to an individual call. The task picks the language that matches the voice.
- `speaking_rate`: how fast to speak, where `1.0` is the normal speed.
- `cache_directory`: where to store the model's compiled kernels and weights, so that the task starts faster next time.

In [ ]:
import tempfile

from mediapipe.tasks.python.audio import text_to_speech
from mediapipe.tasks.python.core import base_options

BaseOptions = base_options.BaseOptions
Delegate = BaseOptions.Delegate

delegate = 'GPU' #@param ['GPU', 'CPU']


def create_tts(delegate):
  options = text_to_speech.TextToSpeechOptions(
      base_options=BaseOptions(model_asset_path=model_path, delegate=delegate),
      default_voice_id='af_heart',
      speaking_rate=1.0,
      cache_directory=os.path.join(tempfile.gettempdir(), 'mediapipe_tts_cache'),
  )
  return text_to_speech.TextToSpeech.create_from_options(options)


try:
  tts = create_tts(Delegate[delegate])
except RuntimeError as e:
  if delegate == 'CPU':
    raise
  print(f'GPU is not available ({e}), falling back to CPU.')
  delegate = 'CPU'
  tts = create_tts(Delegate.CPU)

print(f'TextToSpeech is ready (delegate: {delegate}).')

Each voice has an ID, a display name, and a locale. `list_voices()` returns all voices in the model; here they are grouped by locale.

In [ ]:
from collections import defaultdict

voices = tts.list_voices()
voices_by_locale = defaultdict(list)
for voice in voices:
  voices_by_locale[voice.locale].append(voice.voice_id)

print(f'{len(voices)} voices at {voices[0].sample_rate_hz} Hz:')
for locale, voice_ids in voices_by_locale.items():
  print(f'  {locale:6s} {", ".join(voice_ids)}')

## Synthesizing speech

`synthesize()` converts a piece of text into speech in a single call. The result contains the audio as mono `float32` `samples` in the range `[-1.0, 1.0]`, together with its `sample_rate_hz`. You can play the samples back directly in the notebook. Try a different text or voice on the side of the cell.

In [ ]:
import time

from IPython.display import Audio, display

text = "Hello from MediaPipe Text to Speech, running entirely on device!" #@param {type:"string"}
voice_id = "af_heart" #@param ["af_heart", "af_bella", "am_michael", "am_fenrir", "bf_emma", "bm_george"]

start = time.perf_counter()
result = tts.synthesize(text, voice_id=voice_id)
latency = time.perf_counter() - start

print(f'Synthesized {result.duration_seconds:.2f} s of audio at {result.sample_rate_hz} Hz in {latency * 1000:.0f} ms')
display(Audio(result.samples, rate=result.sample_rate_hz))

To keep the audio, convert the samples to 16-bit integers and write them to a `.wav` file with Python's built-in `wave` module.

In [ ]:
import wave

import numpy as np


def save_wav(path, samples, sample_rate):
  """Saves float32 samples in [-1.0, 1.0] as a 16-bit mono PCM .wav file."""
  pcm = (np.clip(samples, -1.0, 1.0) * 32767).astype(np.int16)
  with wave.open(path, 'wb') as wav_file:
    wav_file.setnchannels(1)
    wav_file.setsampwidth(2)  # 16 bits per sample.
    wav_file.setframerate(sample_rate)
    wav_file.writeframes(pcm.tobytes())


save_wav('tts_output.wav', result.samples, result.sample_rate_hz)
print(f'Saved tts_output.wav ({os.path.getsize("tts_output.wav") / 1000:.0f} kB)')

The result also includes `word_timestamps`: when each word is spoken in the audio, and where it is in the input text (`start_char` to `end_char`). You can use them, for example, to highlight words as they are read aloud.

In [ ]:
for word in result.word_timestamps:
  print(f'{word.start_sec:5.2f}s - {word.end_sec:5.2f}s  {text[word.start_char:word.end_char]!r}')

### Other languages

To speak another language, pick a voice for that language. The task uses the voice's language automatically, so you only need to pass `voice_id`.

In [ ]:
samples_by_voice = {
    'ef_dora': 'Hola, esto es MediaPipe hablando en español.',
    'ff_siwis': 'Bonjour, ceci est MediaPipe qui parle français.',
    'if_sara': 'Ciao, questo è MediaPipe che parla in italiano.',
}

for sample_voice_id, sample_text in samples_by_voice.items():
  sample_result = tts.synthesize(sample_text, voice_id=sample_voice_id)
  print(f'{sample_voice_id}: {sample_text}')
  display(Audio(sample_result.samples, rate=sample_result.sample_rate_hz))

## Streaming synthesis

For longer texts, you don't want to wait until all of the audio has been synthesized before playback starts. `synthesize_stream()` splits the text into clauses and yields each clause's audio as a separate chunk as soon as it is ready. Each chunk has a `chunk_index`, and `is_final` is `True` for the last one.

The most important number here is the time to first audio: how long it takes until the first chunk can start playing. The real-time factor (RTF) is the total processing time divided by the audio duration; values below 1.0 mean that synthesis is faster than playback, so playback never has to wait for the next chunk.

In [ ]:
long_text = (
    'Streaming synthesis splits long text into clauses. '
    'Each clause yields audio as soon as it is ready! '
    'That keeps the time to first audio low, even for long paragraphs.'
)

chunks = []
start = time.perf_counter()
for chunk in tts.synthesize_stream(long_text, voice_id='af_heart'):
  elapsed = time.perf_counter() - start
  words = ' '.join(long_text[w.start_char:w.end_char] for w in chunk.word_timestamps)
  print(f'[{elapsed * 1000:5.0f} ms] chunk {chunk.chunk_index}: {chunk.duration_seconds:.2f} s of audio: {words!r}')
  if chunk.chunk_index == 0:
    time_to_first_audio = elapsed
  chunks.append(chunk)
total_time = time.perf_counter() - start

audio_duration = sum(chunk.duration_seconds for chunk in chunks)
print(f'\nTime to first audio: {time_to_first_audio * 1000:.0f} ms')
print(f'Total: {total_time * 1000:.0f} ms for {audio_duration:.2f} s of audio (RTF: {total_time / audio_duration:.3f})')

# In a real app you would play each chunk as it arrives; here you play them all at once.
display(Audio(np.concatenate([chunk.samples for chunk in chunks]), rate=chunks[0].sample_rate_hz))

### Incremental text input

Sometimes the text itself arrives piece by piece, for example when you read out the response of a language model while it's still being generated. `start_stream()` opens a stream that you can feed with `push_text()`. The stream synthesizes each clause as soon as it is complete, and you read the audio chunks by iterating over the stream. Call `finish_input()` once all text has been pushed, so the stream can synthesize the remaining text and end.

Iterating over the stream waits for the next chunk, so pushing text and reading audio happen in different threads. Below, a background thread simulates a language model that produces one word every 50 ms.

In [ ]:
import threading

generated_text = 'Language models produce text one token at a time. With incremental input, speech can start before the full response is ready.'


def generate_text(stream):
  """Simulates a language model by pushing one word at a time."""
  for word in generated_text.split(' '):
    stream.push_text(word + ' ')
    time.sleep(0.05)
  stream.finish_input()


chunks = []
start = time.perf_counter()
with tts.start_stream(voice_id='af_heart') as stream:
  generator = threading.Thread(target=generate_text, args=(stream,))
  generator.start()
  for chunk in stream:
    print(f'[{(time.perf_counter() - start) * 1000:5.0f} ms] chunk {chunk.chunk_index}: {chunk.duration_seconds:.2f} s of audio')
    chunks.append(chunk)
  generator.join()

display(Audio(np.concatenate([chunk.samples for chunk in chunks]), rate=chunks[0].sample_rate_hz))

Finally, release the model when you are done.

In [ ]:
tts.close()